# Experiments

The changes to the adopted pipeline that were tested, with their results. One rule for every change, fixed before the
results: compared with the adopted version on the same images, a change is adopted only if the 95% bootstrap interval of the
object F1 difference is entirely above 0, with the pixel Dice and the count error not demonstrably worse. Post-processing
changes first had to gain at least +0.005 of out-of-fold F1 on the training images.

1. **Network and training recipe:** recomputed here from the saved checkpoints, on the validation images.
2. **Cross-validation:** read from the saved results (the predictions of its 10 networks are not stored).
3. **Post-processing:** read from the saved results (they need the predictions on the training images, not stored).
4. **Summary.**

Runs on CPU in about 20 minutes. Needs the cleaned masks created by `notebooks/01_eda_green.ipynb`.

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "experiments" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
from evaluation.postprocess import label_instances, postprocess
from evaluation.predict import load_model, predict_probs
from evaluation.tune import bootstrap_diff, counts_row, metrics_from_counts
from train.dataset import GreenEvalDataset, load_img_and_masks, load_split

IMG_DIR = Path("data/green/trainval/images")
MASK_DIR = Path("data/cleaned_masks/trainval/Green/masks")
assert MASK_DIR.exists(), "run notebooks/01_eda_green.ipynb first: it creates the cleaned masks"
pd.set_option("display.width", 250)

## 1. Network and training recipe

Each variant was trained once on the 169 training images (Kaggle, `kaggle/recipe_variants.ipynb` and `kaggle/train.ipynb`),
with the recipe of the adopted network and **one change**, same seed:

| variant | change from the adopted network (c-ResUNet-light) |
|---|---|
| c-ResUNet (Morelli et al., 2021) | with the residual block with 5 × 5 kernels in the bottleneck (receptive field ~160 px instead of ~96) |
| early stopping | the training stops after 15 epochs without improvement of the validation Dice, instead of always 100 epochs |
| no gamma | brightness augmentation with the gain only, without the gamma (contrast) change |
| per-image normalization | each image normalized with its own statistics, instead of the mean and std of the training images |
| wider network | 32 channels in the first level instead of 16 |
| 200 epochs | 200 epochs instead of 100 |

Every network predicts the 43 validation images from its checkpoint (`best.pt`); the predictions go through the basic
post-processing used for these comparisons: threshold 0.5, fill holes, objects < 120 px removed.

In [ ]:
NETWORKS = {   # name → run folder with best.pt and log.csv
    "adopted (c-ResUNet-light)": "runs/cresunet_light",
    "c-ResUNet (Morelli et al., 2021)": "runs/cresunet",
    "early stopping": "runs/cresunet_light_early_stop",
    "no gamma": "runs/cresunet_light_no_gamma",
    "per-image normalization": "runs/cresunet_light_per_image",
    "wider network": "runs/cresunet_light_wide",
    "200 epochs": "runs/cresunet_light_200ep",
}
_, val_names = load_split(Path("splits/green_split.json"))
images, masks = load_img_and_masks(IMG_DIR, MASK_DIR, val_names)

counts, rows = {}, []
for name, run in NETWORKS.items():
    model, mean, std, epoch, norm = load_model(Path(run) / "best.pt", torch.device("cpu"))
    dataset = GreenEvalDataset(images, masks, val_names, mean, std, norm=norm)
    probs, _ = predict_probs(model, dataset, torch.device("cpu"))
    per_image = []
    for prob, mask in zip(probs, masks):
        labels = label_instances(postprocess(prob.astype(np.float32), 0.5))
        per_image.append(counts_row(labels, mask > 0))
    counts[name] = np.array(per_image)
    m = metrics_from_counts(counts[name])
    rows.append({"network": name, "parameters": sum(p.numel() for p in model.parameters()), "best epoch": epoch,
                 "Dice": m["dice"], "F1": m["obj_f1"], "precision": m["obj_precision"], "recall": m["obj_recall"],
                 "count MAE": m["count_mae"]})
    print(f"{name}: done")
print()
print(pd.DataFrame(rows).round(3).to_string(index=False))

Difference "variant − adopted network" for each metric, with its paired 95% bootstrap interval over the 43 images (for the
count error a negative difference is better), and the decision of the rule:

In [ ]:
adopted = counts["adopted (c-ResUNet-light)"]
rows = []
for name in list(NETWORKS)[1:]:
    ci = {}
    for metric in ["obj_f1", "dice", "obj_precision", "obj_recall", "count_mae"]:
        ci[metric] = bootstrap_diff(counts[name], adopted, metric)
    passes = ci["obj_f1"][0] > 0 and ci["dice"][1] >= 0 and ci["count_mae"][0] <= 0
    row = {"variant": name}
    for metric, (low, high) in ci.items():
        row[metric] = f"[{low:+.3f}, {high:+.3f}]"
    row["adopted"] = "yes" if passes else "no"
    rows.append(row)
differences = pd.DataFrame(rows).set_index("variant")
print(differences.to_string())

# check: the F1 intervals are those of notebook 02, section 7
expected = {"c-ResUNet (Morelli et al., 2021)": "[-0.020, +0.006]", "early stopping": "[-0.033, -0.001]",
            "no gamma": "[-0.013, +0.009]", "per-image normalization": "[-0.037, +0.005]",
            "wider network": "[-0.011, +0.019]", "200 epochs": "[-0.005, +0.016]"}
for name, interval in expected.items():
    assert differences.loc[name, "obj_f1"] == interval, name
print("\ncheck: identical to notebook 02")

No variant passes the rule.
- **Early stopping** is demonstrably worse in F1 and precision: the training stops at epoch 52 (best epoch 37), with the
  learning rate still high, before the cosine schedule has reduced it.
- **Per-image normalization** makes the count error demonstrably worse.
- The **wider network** (4× the parameters) has a demonstrably higher precision, but no demonstrable gain in F1; on the
  training images it does not reproduce the labels better either (notebook 02): the capacity of the network is not the limit.
- The **c-ResUNet of Morelli et al.** (2.6× the parameters) brings no demonstrable difference: the block with 5 × 5 kernels
  is not needed on these images.
- **No gamma** and **200 epochs**: no demonstrable difference.

The validation Dice of each variant during the training, next to the adopted network:

In [ ]:
adopted_log = pd.read_csv("runs/cresunet_light/log.csv")
fig, axes = plt.subplots(2, 3, figsize=(16, 7), sharey=True)
for ax, name in zip(axes.ravel(), list(NETWORKS)[1:]):
    log = pd.read_csv(Path(NETWORKS[name]) / "log.csv")
    ax.plot(adopted_log.epoch, adopted_log.val_dice_global, color="grey", label="adopted")
    ax.plot(log.epoch, log.val_dice_global, color="#2a78d6", label=name)
    ax.set_title(name)
    ax.set_xlabel("epoch")
    ax.set_ylim(0.65, 0.78)
    ax.legend(loc="lower right")
for ax in axes[:, 0]:
    ax.set_ylabel("validation Dice")
plt.tight_layout()
plt.show()

The curves agree with the table: per-image normalization stays below the adopted network for the whole second half of the
training; early stopping ends at epoch 52, while the adopted network is still improving; with 200 epochs the validation Dice
does not go higher; the other variants follow the adopted network within the fluctuations between epochs.

## 2. Cross-validation

The 212 trainval images in 5 folds stratified by the number of cells (`train/kfold_split.py`). For each fold, a network is
trained on the other four folds and predicts the held-out fold with its last checkpoint (epoch 100), so every image is
predicted by a network that has not seen it; the object classifier of each fold is chosen and trained on the objects of the
other folds. Two recipes: the adopted one (**L**) and the adopted one + the weight maps of Morelli et al. (2021) in the
loss (**LW**). Executed on Kaggle with `kaggle/cross_validation.ipynb`; the comparison is read from `runs/cv_compare/`.

Names of the pipelines: **ws20** = watershed with seeds 20 px apart, **clf** = object classifier, **a300** = minimum area
300 px (otherwise 120 px).

In [ ]:
rows, all_held_out = [], []
for k in range(5):
    train_names, held_out = load_split(Path(f"splits/green_cv{k}.json"))
    rows.append({"fold": k, "training images": len(train_names), "held-out images": len(held_out)})
    all_held_out += held_out
assert len(all_held_out) == len(set(all_held_out)) == 212      # every image is held out exactly once
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8), sharey=True)
for ax, (recipe, prefix) in zip(axes, [("L: adopted recipe", "cresunet_light_cv"), ("LW: + weight maps", "cresunet_light_wm_cv")]):
    for k in range(5):
        log = pd.read_csv(f"runs/{prefix}{k}/log.csv")
        ax.plot(log.epoch, log.val_dice_global, label=f"fold {k}")
    ax.set_title(recipe)
    ax.set_xlabel("epoch")
    ax.set_ylim(0.6, 0.8)
axes[0].set_ylabel("validation Dice (held-out fold)")
axes[0].legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
cv = json.load(open("runs/cv_compare/results.json"))
pooled = pd.DataFrame(cv["pooled"]).T
print("all 212 images:")
print(pooled[["dice", "obj_f1", "obj_precision", "obj_recall", "n_merged", "count_mae"]].round(3).to_string())

per_fold = pd.read_csv("runs/cv_compare/per_fold.csv")
print("\nobject F1 per fold:")
print(per_fold.pivot(index="variant", columns="fold", values="obj_f1").round(3).to_string())

In [ ]:
comparisons = {"weight maps: LW + ws20 + clf - L + ws20 + clf": cv["primary"]["ci95"],
               "watershed: L + ws20 + clf - L + clf": cv["descriptive"]["L + ws20 + clf − L + clf"],
               "min area 300 px: L + ws20 + a300 + clf - L + ws20 + clf": cv["replication"]["ci95"]}
rows = []
for name, ci in comparisons.items():
    row = {"comparison (212 images)": name}
    for metric in ["obj_f1", "dice", "obj_precision", "obj_recall", "count_mae"]:
        low, high = ci[metric]
        row[metric] = f"[{low:+.3f}, {high:+.3f}]"
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

- **Weight maps:** better in 5 folds of 5, but the F1 difference is not demonstrable on the 212 images and the merged
  objects do not decrease (32 and 32): not adopted.
- **Watershed (20 px):** better in 4 folds of 5 and demonstrable on the 212 images (recall up): part of the adopted pipeline.
- **Minimum area 300 px:** better in 5 folds of 5 and demonstrable (precision up): adopted, after the same result on the
  validation images (notebook 03).
- The differences between folds, i.e. between groups of images (F1 0.77–0.85), are much larger than the differences between
  recipes: folds 0 and 4 are the hardest for both recipes (validation curves above).

## 3. Post-processing

Each variant changes one element of the pipeline of notebook 03, starting from the minimum area of 120 px and seeds 20 px
apart, with the object classifier chosen again for it. "Gain" is the out-of-fold F1 on the 169 training images minus that
of the starting pipeline; a variant is checked on the validation images only if the gain is at least +0.005. These results
need the predictions of the network on the training images, which are not stored (290 MB): they are read from
`runs/cresunet_light/postprocessing_ablations.json` and `runs/cresunet_light/tta_check.json`.

In [ ]:
ablations = json.load(open("runs/cresunet_light/postprocessing_ablations.json"))
p_star = ablations["p_star"]
rows = [{"min area (px)": 120, "seeds at least (px)": 20, "merged": p_star["merged_train"],
         "cells cut in two": p_star["split_cells_train"], "out-of-fold F1": p_star["oof_f1"], "gain": 0.0,
         "validation: F1 difference": "(starting pipeline)"}]
for entry in ablations["grid"]:
    checked = "not checked"
    if "val_ci" in entry:
        low, high = entry["val_ci"]["obj_f1"]
        checked = f"[{low:+.3f}, {high:+.3f}]"
    rows.append({"min area (px)": entry["min_area"], "seeds at least (px)": entry["min_distance"],
                 "merged": entry["merged"], "cells cut in two": entry["split_cells"], "out-of-fold F1": entry["oof_f1"],
                 "gain": entry["gain"], "validation: F1 difference": checked})
print(pd.DataFrame(rows).round(4).to_string(index=False))

In [ ]:
tta = json.load(open("runs/cresunet_light/tta_check.json"))
low, high = tta["ci95_tta_minus_network"]["obj_f1"]
print(f"test-time augmentation (training images, network alone): F1 {tta['network']['obj_f1']:.4f} -> "
      f"{tta['network + TTA']['obj_f1']:.4f}, gain {tta['f1_gain']:+.4f} (required +0.005), 95% CI [{low:+.4f}, {high:+.4f}]")
print(f"features from the annotation protocol: out-of-fold gain {ablations['protocol']['gain']:+.5f}")

rows = []
for entry in ablations["stacking"]:
    rows.append({"candidate cells from": entry["proposer"], "candidates": entry["candidates"],
                 "of which annotated cells": entry["true"], "ceiling gain": entry["ceiling_gain"],
                 "out-of-fold gain": entry.get("gain", "not tried (ceiling < 0.02)")})
print()
print(pd.DataFrame(rows).round(4).to_string(index=False))

print("\nmorphological closing, pairs of predicted objects within 6 px (training images):", ablations["closing_pairs_train"])

- **Minimum area:** 300 px is the smallest limit with a gain of at least +0.005 and a demonstrable improvement on the
  validation images (adopted); 350 and 400 px gain as much out of fold but not demonstrably on the validation images.
- **Watershed distance:** 12 and 15 px cut single cells in two, and their gains are below +0.005; without watershed or at
  25 px more merged objects remain: 20 px is kept.
- **Test-time augmentation** (average of the 8 orientations of each image): gain +0.0048, just below the required +0.005.
- **Features from the annotation protocol** (roundness, dark nucleoli, edge sharpness, size, texture): no gain; they repeat
  the information of the existing features.
- **Candidate cells from other methods** (robust threshold, Cellpose zero-shot), added through a second classifier: the
  ceiling (all true candidates added) would be up to +0.048, but the classifier cannot separate the few true candidates
  from the many false ones (gain ≤ +0.0002).
- **Morphological closing** (to repair fragmented cells): within 6 px there is 1 pair of pieces of the same cell against
  18 pairs of different cells: it would mostly join different cells.

## 4. Summary

| change | result | decision |
|---|---|---|
| c-ResUNet of Morelli et al. (5 × 5 block) | no demonstrable difference, 2.6× the parameters | not adopted |
| early stopping | F1 demonstrably worse | not adopted |
| no gamma | no demonstrable difference | not adopted |
| per-image normalization | count error demonstrably worse | not adopted |
| wider network | precision up, F1 not demonstrable | not adopted |
| 200 epochs | no demonstrable difference | not adopted |
| weight maps (cross-validation) | F1 not demonstrable, merges unchanged | not adopted |
| watershed, seeds 20 px apart (cross-validation) | F1 demonstrably better | **adopted** |
| minimum area 300 px (validation and cross-validation) | F1 demonstrably better | **adopted** |
| other minimum areas, other watershed distances | gain below +0.005, or not demonstrable on validation | not adopted |
| test-time augmentation | gain +0.0048, below +0.005 | not adopted |
| features from the annotation protocol | no gain | not adopted |
| candidate cells from other methods | gain ≤ +0.0002 | not adopted |
| morphological closing | would join different cells | not adopted |